# 07 · Summary statistics

From τ of several lines, `DeltaFieldBuilder` forms, per skewer, the overflux δ_j = F_j/⟨F_j⟩ − 1 of every line, the
products δ_A of every line subset and the total δ_tot = ∏F_j/⟨∏F_j⟩ − 1, which decomposes **exactly** into the
subset terms (notes `sec_flux.tex`, Sec. Overflux). `OverfluxStatistics1D` gives P1D and ξ1D (an exact discrete
Fourier pair; P_ab = (dv/N) F_a F_b^*, notes `subsec_P1D.tex`) of the total, of every line pair and of every order
|A| + |B|; `OverfluxStatistics3D` does the same on a grid of skewers. Runtime: ~30 s.

**Requires:** τ (06) and the lines (01) for the reference frame; sections 2–5 build τ directly from toy fields.  
**Independent of:** simulation outputs (04).

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import (use_style, ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, LABELS, FIGSIZES,
                                      add_panel_letters, plot_dag, plot_spectra, zoom_inset)
use_style()                                        # the dark, colour-blind-safe style of every figure (skewerjax.utils.plotting)
%matplotlib inline

In [ ]:
plot_dag(('overflux', 'stats')); plt.show()

## 1 · From the integrator's output to δ_F, step by step

The integrator returns one τ per line, each in its **own** line-centred frame. Four steps lead to the overflux:
(a) F_j = e^{−τ_j} per line; (b) every line moved onto the reference (Lyα) frame by `wrap_to_ref_line`, a shift by
−Δv_j that wraps around the box; (c) the total F = ∏_j F_j; (d) the mean flux ⟨F⟩ and δ_F = F/⟨F⟩ − 1, per line and for
the total (`DeltaFieldBuilder`). The toy box of notebook 00 (50 cMpc, z = 2.2, one GRF for H I and Si III), now
integrated as a **periodic** box: `velocity_period = P`, the box length in velocity, and a pixel grid that covers
exactly one period (as `SimulationSkewers.spectral_grid`).

In [ ]:
from skewerjax.core import (LineBundle, Cosmology, SkewerBundle, grid_skewer_points, phys, Fields, ScalarField, VectorField,
                            PowerLawGRF, GaussianDensity, LogRamp, VelocityRamp, Constant, compute_optical_depth, wrap_to_ref_line)
from skewerjax.utils.plotting import plot_raw_to_overflux
lines2 = LineBundle(["HI-LyA", "SiIII-1206"], ref_line_id="HI-LyA")
eds = Cosmology(H0=phys.H0, Omega_r_0=0.0, Omega_m_0=1.0, Omega_k_0=0.0, Omega_lambda_0=0.0)
Lb, z = 50.0, 2.2
pts = grid_skewer_points(Lb, 8, 8193, direction="y").reshape(64, 8193, 3).astype(jnp.float64)   # first and last point: the same place
sbq = SkewerBundle(spatial_points=pts, cosmologies=eds, a0=1 / (1 + z), x0=Lb)
a_d = 1 / (1 + z)
P = float(phys.c_kms * jnp.log(a_d / sbq.scale_factors[0, 0, 0]))              # the box in velocity: one period [km/s]
N_v = 2048
dvq = P / N_v
vq = (np.arange(N_v) + 0.5) * dvq                                                # one period, N_v dv = P exactly
grf_q = PowerLawGRF(A_s=4.0, k_0=2.0, n_s=-1.25, L=(Lb,) * 3, N=(64,) * 3)
key_q = jax.random.PRNGKey(0)
dens_q = {"HI": GaussianDensity(key_q, grf_q, n0_min=1e-16, n0_max=1.35e-11, contrast=1.2),
          "SiIII": GaussianDensity(key_q, grf_q, n0_min=1e-16, n0_max=1.35e-11 / 250, contrast=1.2)}
toy_sp = lambda d: {"temperature": ScalarField(LogRamp(1.5e4, 1.5e4, Lb)), "velocity": VectorField(VelocityRamp(0.0, 0.0, Lb)),
                    "number_density": ScalarField(d), "turbulence": ScalarField(Constant(0.0)), "collisional": ScalarField(Constant(0.0))}
toy = Fields({sp: toy_sp(d) for sp, d in dens_q.items()})
tau_own = compute_optical_depth(sbq, vq, lines2, toy, a_dagger=a_d, velocity_period=P)     # periodic: every point through its nearest image
tau_ref = wrap_to_ref_line(tau_own, vq, lines2)
dV = float(lines2.delta_v[1])
print(f"P = {P:.1f} km/s, dv = {dvq:.3f} km/s, Delta v(Si III) = {dV:.1f} km/s = {dV / P:+.3f} P | <F>: Lya {np.exp(-np.asarray(tau_ref[0])).mean():.4f}, "
      f"Si III {np.exp(-np.asarray(tau_ref[1])).mean():.4f}, total {np.exp(-np.asarray(tau_ref)).prod(0).mean():.4f}")
fig, axs = plot_raw_to_overflux(vq, tau_own, tau_ref, ["HI-LyA", "SiIII-1206"], skewer=0, period=P)
fig.suptitle("From the integrator's raw output to the overflux (skewer 0 of 64)", fontsize=22); plt.show()

**Periodicity.** The two integrals the core offers, on four of the skewers: the direct one
(`compute_optical_depth` default, `velocity_period=None`) and the periodic one. (a) They differ only near the ends of
the grid, where the direct integral loses the absorption of gas whose line spills over the box edge. (b) On the
periodic integral the wrap is exact: Si III shifted by `wrap_to_ref_line` equals Si III evaluated directly at
v + Δv in its own frame, up to the linear interpolation between pixels (Δv is not a multiple of dv). (c) Wrapping the
direct integral moves its cut edge into the middle of the reference-frame spectrum.

In [ ]:
sb4 = SkewerBundle(spatial_points=pts[:4], cosmologies=eds, a0=a_d, x0=Lb)
tau_dir = compute_optical_depth(sb4, vq, lines2, toy, a_dagger=a_d)                                        # direct (default)
si_only = LineBundle(["SiIII-1206"], ref_line_id="SiIII-1206")
tau_si_shift = compute_optical_depth(sb4, vq + dV, si_only, Fields({"SiIII": toy_sp(dens_q["SiIII"])}), a_dagger=a_d, velocity_period=P)[0]
F_per, F_dir = np.exp(-np.asarray(tau_own[:, :4])), np.exp(-np.asarray(tau_dir))
F_wrap_dir = np.exp(-np.asarray(wrap_to_ref_line(tau_dir, vq, lines2)))
fig, axs = plt.subplots(1, 3, figsize=FIGSIZES["triple"], layout="constrained")
for k in range(4):
    axs[0].plot(vq, F_dir[0, k] - F_per[0, k], color=list(TOL.values())[k], label=f"skewer {k}")
    axs[1].plot(vq, np.exp(-np.asarray(tau_ref[1, k])) - np.exp(-np.asarray(tau_si_shift[k])), color=list(TOL.values())[k])
    axs[2].plot(vq, F_wrap_dir[1, k] - np.exp(-np.asarray(tau_ref[1, k])), color=list(TOL.values())[k])
axs[0].set(xlabel=LABELS["v_dagger"], ylabel=r"$F_{\rm direct} - F_{\rm periodic}$ (Ly$\alpha$)", title="direct vs periodic integral")
axs[1].set(xlabel=LABELS["v_ref"], ylabel=r"$\Delta F$ (Si III)", title=r"wrap vs direct evaluation at $v + \Delta v$")
axs[2].set(xlabel=LABELS["v_ref"], ylabel=r"$\Delta F$ (Si III)", title="wrap of the direct integral − wrap of the periodic one")
axs[2].axvline(np.mod(-dV, P), color=FG, ls=":", lw=1.2)
axs[0].legend(fontsize=12); add_panel_letters(axs); plt.show()
print(f"max |dF|: (a) {np.abs(F_dir - F_per).max():.2e} | (b) {np.abs(np.exp(-np.asarray(tau_ref[1, :4])) - np.exp(-np.asarray(tau_si_shift))).max():.2e} "
      f"| (c) {np.abs(F_wrap_dir[1] - np.exp(-np.asarray(tau_ref[1, :4]))).max():.2e} (the dotted line: where the cut edge lands, -Delta v mod P)")

**Q1 — how periodicity is handled, and whether it is correct.**
- **Sampled fields are periodic in space.** `GaussianDensity` wraps every position into the box and interpolates the
  grid with periodic indices (`core/toy_fields.py` L353–L357, `_trilinear_periodic_sample` L291); the simulation
  deposition takes every mesh-generating point at its minimum image (`simulations/deposition.py`, module docstring,
  "Periodicity"). A skewer from 0 to L with both end points (as here) closes on itself; the trapezoid line elements give
  the two end points half a weight each (`core/skewers.py` L15–L23), i.e. the rectangle rule of a closed path.
- **The flux, core integrator.** The default `velocity_period=None` is the direct integral of the finite path. With
  `velocity_period=P`, `frequency_offset_GHz` replaces every detuning by its minimum image in [−P/2, P/2)
  (`core/spectral.py` L119–L121): each point absorbs through its nearest periodic image, and the box behaves as a
  periodic universe in velocity.
- **The flux, simulations.** `SimulationSkewers` always integrates with the period of its frame
  (`IntegrationOptions.periodic=True`, `simulations/snapshot_skewers.py` L81, L344–L373; `absorbers.tau_absorbers` passes
  `velocity_period=frame.period_kms`, L308) on `spectral_grid`, which covers exactly one period (L203–L217).
- **One axis for all lines.** `wrap_to_ref_line` (`core/optical_depth.py` L351–L365) shifts line j by Δv_j with periodic
  interpolation, period N_v dv. It is exact when τ_j has that same period, panel (b). That holds by construction for
  simulations; on the core path it holds only if the user integrates with `velocity_period = N_v dv` on a grid of N_v
  pixels covering one period, as above. The quickstart (00) integrates directly on `linspace(0, v_max, 512)`, so its
  wrapped spectra carry the cut edge of panel (c), and its grid has N_v dv = v_max · 512/511, one pixel longer than the
  box (an open decision of the reply; the behaviour is not changed here).
- **fake_spectra.** `absorption.cpp:add_tau_particle` (L243–L280) adds every cell to the pixels from its own up to
  ±nbins/2 away and wraps the pixel index modulo nbins: the same nearest-image rule (both codes cut damping wings
  half a box away). Every line is computed on its own velocity axis of the box (`spectra.py` `get_tau` L918); there is
  no shift onto a reference line (`get_observer_tau` L932 picks one line per ion). Moving Si III onto the Lyα frame is a
  SkewerJAX step.
- **Consequences.** On the periodic grid k_n = 2πn/P, cos(k_n Δv) = cos(k_n (Δv mod P)), so the P1D of the wrapped
  spectra is exactly that of a shifted copy: the McDonald oscillation is sampled correctly (it only looks aliased when
  P < Δv). ξ1D peaks at Δv mod P (and P − that), not at Δv. Physically, the Si III that lands on a Lyα pixel comes from
  gas Δv mod P away in the same box instead of Δv away: for boxes shorter than Δv the Lyα × Si III correlation is that of
  a smaller separation, a finite-box effect common to every periodic simulation.

In [ ]:
from skewerjax.core import PowerLawGRF, DeltaFieldBuilder
from skewerjax.core.stats import OverfluxStatistics1D, OverfluxStatistics3D, fit_mcdonald_a, fit_ma_model_profiled, mcdonald_factor, ma_factor
from skewerjax.utils.plotting import plot_orders_superimposed, plot_pk_recovery, hybrid_k_edges, logbin
N_sk, N_v, Lv = 400, 1000, 1000.0                         # skewers, pixels, velocity length [km/s]
dv, shift = Lv / N_v, 100                                  # the second line sits 100 km/s away
grf = PowerLawGRF(1.0, 2 * np.pi / 100, 0.5, L=Lv, N=N_v)
grf = PowerLawGRF(1.0 / float(grf.variance()), 2 * np.pi / 100, 0.5, L=Lv, N=N_v)     # sigma_delta = 1
delta = jax.vmap(grf.sample)(jax.random.split(jax.random.PRNGKey(7), N_sk))
tau = jnp.stack([jnp.exp(delta), 0.5 * jnp.exp(jnp.roll(delta, shift, axis=-1))])        # line 2: same gas, shifted, weaker
ov = DeltaFieldBuilder(tau, labels=["1", "2"], keep_fields=True).build()
print("subset keys:", ov.subset_keys, "| C_s of the first skewers:", np.round(np.asarray(ov.C_per_sample[:3, 0]), 4))
fig, axs = plt.subplots(2, 1, figsize=(14, 7), sharex=True, layout="constrained")
vv = np.arange(N_v) * dv
for j, lab in enumerate(ov.labels):
    axs[0].plot(vv, ov.fields[j, 0], color=list(TOL.values())[j], label=f"$F_{lab}$"); axs[1].plot(vv, ov.delta_terms[lab][0], color=list(TOL.values())[j])
axs[1].plot(vv, ov.total[0], color=FG, lw=1.2, label=r"$\delta_{\rm tot}$")
axs[0].set(ylabel="F", xlim=(0, 300)); axs[1].set(xlabel="v [km/s]", ylabel=r"$\delta$"); axs[0].legend(); axs[1].legend()
add_panel_letters(axs); fig.suptitle("Two lines of the same gas: the second is a shifted, weaker copy", fontsize=22); plt.show()

## 2 · P1D and ξ1D by order

Order n collects the terms (A, B) with |A| + |B| = n (order 0 is the normalisation term 𝒞). Their sum is the total to machine precision.

In [ ]:
st = OverfluxStatistics1D(ov, dv).compute_p1d().compute_xi1d()
fig, axs = plot_orders_superimposed(st, "P", figsize=(12, 8))
fig.suptitle("P1D: orders 2, 3, 4 add up to the total", fontsize=20)
fig, axs = plot_orders_superimposed(st, "xi", delta_v_list=[0.0, shift * dv], max_lag=300.0, figsize=(12, 8))
fig.suptitle(r"$\xi_{\rm 1D}$: the cross term peaks at the line separation (dashed)", fontsize=20); plt.show()

### Each order, term by term

The same decomposition one level down: every term (A, B) of order |A| + |B| = n, with (A, B) and (B, A) drawn
together as A–B (their sum, which is real). 𝒞 is the order-0 normalisation term. Order 2 holds the Gaussian-like
products (1–1, 1–2, 2–2) and the normalisation cross term 𝒞–12; orders 3 and 4 are the non-linear corrections of
the product of the two fluxes (after the pre-refactor `8_P1D_xi1D`).

In [ ]:
from skewerjax.utils.plotting import plot_terms_by_order
fig, axs = plot_terms_by_order(st, "P")
fig.suptitle("P1D: every order split into its terms; the terms add up to the order (white)", fontsize=20); plt.show()
fig, axs = plot_terms_by_order(st, "xi", max_lag=300.0)
for ax in axs:
    ax.axvline(shift * dv, color=FG, ls=":", lw=1.2)
fig.suptitle(r"$\xi_{\rm 1D}$ by order and term: the cross terms 1–2 carry the line separation (dotted)", fontsize=20); plt.show()
P_terms = {n: sum(np.asarray(st.Pk_total_from_terms_contributions[p]) for p in st.term_pairs if st.pair_orders[p] == n) for n in st.Pk_total_by_order}
print("max |sum of the terms - order| / max|total| per order:",
      {n: f"{np.max(np.abs(P_terms[n] - np.asarray(st.Pk_total_by_order[n]))) / np.max(np.abs(np.asarray(st.Pk_total))):.1e}" for n in P_terms})

## 3 · Recovering an input spectrum, and the metal models

In the linear regime (σ_δ ≪ 1) the total P1D of a line plus a weaker copy at Δv is τ0²[1 + a² + 2a cos(kΔv)]P_in,
McDonald et al.'s Si III model. `fit_mcdonald_a` recovers a; Ma et al.'s model (`fit_ma_model_profiled`) adds
k-dependent amplitudes and contains McDonald's as a limit.

In [ ]:
a_in = 0.5
g_lin = PowerLawGRF(1e-4 * grf.A_s, grf.k_0, grf.n_s, L=Lv, N=N_v)
d = jax.vmap(g_lin.sample)(jax.random.split(jax.random.PRNGKey(8), N_sk))
st_l = OverfluxStatistics1D(DeltaFieldBuilder(jnp.stack([jnp.exp(d), a_in * jnp.exp(jnp.roll(d, shift, -1))])).build(), dv).compute_p1d()
k = np.asarray(st_l.k)[1:]; P_tot, P_11 = np.asarray(st_l.Pk_total)[1:], np.asarray(st_l.Pk_pair[("1", "1")])[1:]
fit = fit_mcdonald_a(k, P_tot, P_11, delta_v=shift * dv, k_range=(0.0, 1.0))
fma = fit_ma_model_profiled(k, P_tot, P_11, delta_v=shift * dv, k_range=(0.0, 1.0), run_global=False)
edges = hybrid_k_edges(2 * np.pi / Lv, float(k[-1]))
fig, axs = plot_pk_recovery(st_l.bin_p1d(edges), st_l.bin_p1d(edges, Pk=np.concatenate([[0], mcdonald_factor(k, a_in, shift * dv) * np.asarray(g_lin.power(k))]))[0],
                            label=r"$P_{\rm tot}$, 400 skewers", figsize=(13, 9))
axs[0].plot(k, mcdonald_factor(k, a_in, shift * dv) * np.asarray(g_lin.power(k)), color=FG, label="McDonald, input a")
axs[0].legend(); fig.suptitle(f"Linear regime: P_tot follows McDonald's form; fitted a = {fit['a']:.4f} (input {a_in})", fontsize=20)
print(f"McDonald chi2 = {fit['chi2']:.3e} | Ma et al.: a_auto = {fma['a_auto']:.3f}, a_cross = {fma['a_cross']:.3f}, chi2 = {fma['chi2']:.3e}"); plt.show()

## 4 · Input spectrum against the measured P1D (Q2)

The GRF of section 1 has an input power law P3D(k) = A_s (k/k_0)^{n_s−1}. Along a line of the grid, sampled at the grid
nodes, its expected 1D spectrum is the discrete projection P1D(k_∥) = L_⊥⁻² Σ_{k_⊥} P3D(√(k_∥² + k_⊥²))
(`plotting.grf_p1d_along_axis`, same normalisation as `OverfluxStatistics1D`). Three levels are compared with it, on
the 64 skewers of section 1 (k in s/km through the Hubble flow of the box, dv/dx = P/L):
1. **the field itself:** δ on the 64 × 64 grid lines (exact nodes), and δ along the skewers (trilinear interpolation,
   recovered exactly from n by inverting the density mapping);
2. **the number densities:** δ_n = n/⟨n⟩ − 1 of H I, of Si III and of their sum;
3. **the flux:** δ_F of Lyα alone, of Si III alone and of the total, on the Lyα frame.

In [ ]:
from types import SimpleNamespace
from skewerjax.utils.plotting import grf_p1d_along_axis, plot_p1d_input_vs_measured, logbin, pk_sigma
u = P / Lb                                                           # km/s per cMpc of the box (Hubble flow)
one = lambda d: SimpleNamespace(subset_keys=["a"], delta_terms={"a": d}, C_per_sample=jnp.zeros((d.shape[0], 1)), total=d)
p1d = lambda d, dx: OverfluxStatistics1D(one(jnp.asarray(d)), dx).compute_p1d()
dH = dens_q["HI"]
var_q = float(grf_q.variance())
# 1. the field: grid lines along y at the nodes, and along the skewers (inverting log10 n = mid + amp tanh(delta / contrast))
d_nodes = np.moveaxis(np.asarray(dH.delta_grid), 1, -1).reshape(-1, 64)
st_nodes = p1d(d_nodes, Lb / 64)
qv = sbq.quadri_vectors[0][:, :-1]                                   # drop the closing point: 8192 points per skewer
n_sk = np.asarray(toy.number_density(qv))                            # (2, 64, 4096)
d_skew = dH.contrast * np.arctanh((np.log10(n_sk[0]) - float(dH.log_mid)) / float(dH.log_amp))
st_skew = p1d(d_skew, Lb / 8192)
# 2. the number densities, 3. the flux (Lya frame)
dn = lambda n: n / n.mean() - 1.0
st_n = {lab: p1d(dn(n), Lb / 8192) for lab, n in (("H I", n_sk[0]), ("Si III", n_sk[1]), ("H I + Si III", n_sk.sum(0)))}
F_ref = np.exp(-np.asarray(tau_ref))
st_F = {lab: p1d(F / F.mean() - 1.0, dvq) for lab, F in (("Lyα alone", F_ref[0]), ("Si III alone", F_ref[1]), ("total", F_ref.prod(0)))}
print(f"grid Nyquist k_N = {np.pi * 64 / Lb / u:.4f} s/km | 64 x 64 node lines, 64 skewers x 8192 points | input variance of delta/sigma = 1")

In [ ]:
nmax = 32                                                            # the modes of the 64^3 grid: k_n = 2 pi n / P, n <= N/2
kk = 2 * np.pi * np.arange(1, nmax + 1) / P                           # [s/km]
Pin, sig_nodes = (x / var_q * u for x in grf_p1d_along_axis(grf_q, kk * u, axis=1, return_sigma=True))   # input projected to 1D [km/s]
take = lambda st, scale=u: np.asarray(st.Pk_total)[1:nmax + 1] * scale
err = lambda st, scale=u: np.asarray(st.Pk_total_err)[1:nmax + 1] * scale              # scatter between skewers / sqrt(N)
curves_n = {"δ at the grid nodes": (take(st_nodes), sig_nodes, FG, dict(ls="none", marker="o", ms=7)),
            "δ along the skewers (interpolated)": (take(st_skew), err(st_skew), TOL["cyan"], dict(lw=2.2, ls="--"))}
for lab, c, ls in (("H I", EC["HI"], "-"), ("Si III", EC["SiIII"], "-"), ("H I + Si III", TOL["green"], ":")):
    curves_n[f"δ_n {lab}"] = (take(st_n[lab]), err(st_n[lab]), c, dict(lw=2.4 if ls == ":" else 2.0, ls=ls))
curves_F = {f"δ_F {lab}": (take(st_F[lab], 1.0), err(st_F[lab], 1.0), c, dict(lw=2.0, ls=ls))
            for lab, c, ls in (("Lyα alone", EC["HI-LyA"], "-"), ("Si III alone", EC["SiIII-1206"], "-"), ("total", FG, "--"))}
fig, axs = plt.subplots(2, 2, figsize=(24, 12), sharex=True, gridspec_kw={"height_ratios": [2.2, 1.3]}, layout="constrained")
plot_p1d_input_vs_measured(kk, curves_n, Pin, ref_label="input (projected)", axs=axs[:, 0])
plot_p1d_input_vs_measured(kk, curves_F, Pin, ref_label="input (projected)", axs=axs[:, 1])
axs[0, 0].set_title("the field and the number densities", fontsize=18); axs[0, 1].set_title("the flux (Lyα frame)", fontsize=18)
add_panel_letters(axs.ravel()); fig.suptitle("Input power law (projected to 1D) against the P1D measured at every level, mode by mode", fontsize=22)
plt.show()
lo = slice(0, nmax // 4)                                              # k < k_N / 4
for lab, (Pm, *_rest) in {**curves_n, **curves_F}.items():
    r = Pm[lo] / Pin[lo]
    print(f"{lab:36s} P / input, n = 1..{nmax // 4}: mean {r.mean():9.3g}, range [{r.min():.3g}, {r.max():.3g}]; at k_N: {Pm[-1] / Pin[-1]:.3g}")
pull = (take(st_nodes) - Pin) / sig_nodes
pull_g = (take(st_nodes) - Pin) / pk_sigma(Pin, np.asarray(st_nodes.n_modes)[1:nmax + 1])
print(f"delta at the nodes: chi2/dof = {np.mean(pull ** 2):.2f} over {nmax} modes, max |pull| = {np.abs(pull).max():.1f} with the exact sigma "
      f"(sqrt(sum P3D^2) / L_perp^2); {np.mean(pull_g ** 2):.2f} with P sqrt(2 / n_modes), which ignores that few k_perp modes dominate")
pull_s = ((take(st_skew) - Pin) / err(st_skew))[lo]
print(f"delta along the skewers, n = 1..{nmax // 4}: chi2/dof = {np.mean(pull_s ** 2):.2f} with the scatter between the 64 skewers / 8, an "
      "underestimate: the skewers share the low-k_perp modes of one realisation")

**Q2 — is the input recovered?** At the level of the field, yes: δ at the grid nodes follows the projected input
mode by mode (χ² printed above, with the exact error of one realisation), so the GRF and the P1D estimator agree.
Along the skewers the trilinear interpolation between nodes is a low-pass filter: the ratio falls towards the grid
Nyquist k_N. The **number densities do not** recover it: n = 10^{mid + amp tanh(δ/1.2)} spans five decades, so
δ_n = n/⟨n⟩ − 1 is dominated by the density peaks; its P1D is several times the input at low k and has another shape,
and H I and Si III differ because their log amplitudes differ (`GaussianDensity`, `core/toy_fields.py` L328–L357). The
sum H I + Si III is H I (Si III is 250 times less abundant). **The flux** then reshapes it again: F = e^{−τ} saturates
in the troughs, so δ_F has about a tenth of the input power at low k here, falling towards k_N (thermal broadening and
saturation); Si III alone, a weak line (⟨F⟩ ≈ 0.99), carries ~10⁻⁴ of it, and the total equals Lyα within the scatter.

## 3D: P3D and ξ3D on a grid of skewers

The skewers of a regular transverse grid (row-major, N_sk = N_x N_y) give P3D(k_⊥, k_∥) and ξ3D(r_⊥, π); transverse
axes in cMpc, the line of sight in km/s (or anything else the user passes).

In [ ]:
g3 = PowerLawGRF(1.0, 1.0, -1.0, L=(40.0, 40.0, 40.0), N=(32, 32, 64))
d3 = g3.sample(jax.random.PRNGKey(3))
st3 = OverfluxStatistics3D(DeltaFieldBuilder(jnp.exp(0.3 * d3).reshape(1, 32 * 32, 64)).build(), 40.0 / 64, 40.0 / 32).compute_xi3d()
Pb, kperp, _, kpar, _ = st3.bin_p3d_kperp(5)
xb, rperp, _, pi, _ = st3.bin_xi3d_rperp(12)
from skewerjax.utils.plotting import tol_cmap
fig, axs = plt.subplots(1, 2, figsize=(20, 7), layout="constrained")
cm = tol_cmap("iridescent")
for i in range(len(kperp)):
    axs[0].loglog(np.asarray(kpar)[1:], np.asarray(Pb)[i, 1:], color=cm(0.25 + 0.7 * i / len(kperp)), label=f"$k_\perp$ = {float(kperp[i]):.2f}")
axs[0].set(xlabel=r"$k_\parallel$ [1/cMpc]", ylabel=r"$P_{\rm 3D}$", title=r"P3D in bins of $k_\perp$ [1/cMpc]"); axs[0].legend(fontsize=12)
lim = float(np.nanmax(np.abs(np.asarray(xb))))
im = axs[1].pcolormesh(np.asarray(pi), np.asarray(rperp), np.asarray(xb), cmap=tol_cmap("BuRd"), vmin=-lim, vmax=lim, shading="auto")
fig.colorbar(im, ax=axs[1], label=r"$\xi_{\rm 3D}$"); axs[1].set(xlabel=r"$\pi$ [cMpc]", ylabel=r"$r_\perp$ [cMpc]", title=r"$\xi_{\rm 3D}(r_\perp, \pi)$")
add_panel_letters(axs); fig.suptitle("3D statistics of a lognormal field on 32 × 32 skewers", fontsize=22); plt.show()

All the statistics kernels (`p1d_kernel`, `p3d_kernel`, `xi_from_power`, `binned_mean`) are jitted; recovery and exactness checks are in `scripts/validation/check_summary_statistics.py`.

## Summary

- The overflux of a product of lines decomposes exactly into subset terms. P1D and ξ1D split by pair and by order,
  and the orders add up to the total to rounding.
- In the linear regime the total P1D follows McDonald's form, and `fit_mcdonald_a` recovers the amplitude.
- Every line is integrated in its own frame and moved onto the Lyα frame by a periodic shift; the box must
  be periodic in velocity for that shift to be exact (section 1).
- Checks: `scripts/validation/check_summary_statistics.py`.